In [1]:
import pandas as pd
import numpy as np
import os
import glob # 用于查找文件

# --- 1. 定义文件路径 ---
# (.. 返回上一级)
raw_data_dir = '../data/raw/kaggle_data/'
processed_dir = '../data/processed/'
processed_file_path = os.path.join(processed_dir, 'cleaned_PRSA_Data_ALL_STATIONS.csv')

# 确保 'processed' 文件夹存在
os.makedirs(processed_dir, exist_ok=True)

# 查找所有 PRSA_Data...csv 文件
all_csv_files = glob.glob(os.path.join(raw_data_dir, "PRSA_Data_*.csv"))

if not all_csv_files:
    print(f"错误: 在 '{raw_data_dir}' 中没有找到任何 PRSA_Data_*.csv 文件。")
    # 如果出错，后续代码将不会执行
    raise FileNotFoundError

print(f"找到了 {len(all_csv_files)} 个站点的数据文件。")

# --- 2. 循环加载、清洗并合并 ---
all_stations_df_list = []

for filepath in all_csv_files:
    station_name = os.path.basename(filepath).split('_')[2]
    print(f"--- 正在处理: {station_name} ---")
    
    df = pd.read_csv(filepath)
    
    # [步骤 3.1] 创建 datetime 索引
    df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
    df.set_index('datetime', inplace=True)
    
    # [步骤 3.2] 丢弃无用列 (注意: 我们保留 'station'!)
    df.drop(['No', 'year', 'month', 'day', 'hour'], axis=1, inplace=True)

    # [步骤 3.3] 填充缺失值 (NaN)
    # 填充数值型列 (线性插值)
    numeric_cols = df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        df[col] = df[col].interpolate(method='linear')

    # 填充 'wd' (风向)
    df['wd'] = df['wd'].ffill().bfill() # ffill 之后再 bfill 确保万无一失
    
    all_stations_df_list.append(df)

print("\n--- 所有文件处理完毕, 正在合并... ---")

# --- 3. 合并所有 DataFrame ---
# (axis=0, 垂直堆叠)
master_df = pd.concat(all_stations_df_list, axis=0)

# --- 4. 对类别型数据进行 One-Hot 编码 ---
# 现在我们需要对 'wd' 和 'station' 两列进行编码
print("正在对 'wd' 和 'station' 列进行 One-Hot 编码...")
master_df = pd.get_dummies(master_df, columns=['wd', 'station'], prefix=['wd', 'station'])

# --- 5. 保存处理后的数据 ---
master_df.to_csv(processed_file_path)

print("\n--- 多站点数据清理完成! ---")
print(f"总数据量 (行, 列): {master_df.shape}")
print(f"已成功保存清理后的数据到: {processed_file_path}")
print("\n清理后数据的信息:")
master_df.info()
print("\n清理后数据的前5行:")
print(master_df.head())

找到了 12 个站点的数据文件。
--- 正在处理: Aotizhongxin ---
--- 正在处理: Changping ---
--- 正在处理: Dingling ---
--- 正在处理: Dongsi ---
--- 正在处理: Guanyuan ---
--- 正在处理: Gucheng ---
--- 正在处理: Huairou ---
--- 正在处理: Nongzhanguan ---
--- 正在处理: Shunyi ---
--- 正在处理: Tiantan ---
--- 正在处理: Wanliu ---
--- 正在处理: Wanshouxigong ---

--- 所有文件处理完毕, 正在合并... ---
正在对 'wd' 和 'station' 列进行 One-Hot 编码...

--- 多站点数据清理完成! ---
总数据量 (行, 列): (420768, 39)
已成功保存清理后的数据到: ../data/processed/cleaned_PRSA_Data_ALL_STATIONS.csv

清理后数据的信息:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 420768 entries, 2013-03-01 00:00:00 to 2017-02-28 23:00:00
Data columns (total 39 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   PM2.5                  420768 non-null  float64
 1   PM10                   420768 non-null  float64
 2   SO2                    420768 non-null  float64
 3   NO2                    420746 non-null  float64
 4   CO                     420768 non-null  floa